# Flight Delay Prediction  Phase 2
## Predictive Modeling, Hyper-parameter Tuning & Submissions

Loads the same cleaned dataset as Phase 1, then runs tuned classification and regression models and writes Kaggle-style submission CSVs.

## 1. Load & prepare

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd

CSV_PATH = "cleaned_final_train.csv"   # the cleaned dataset shipped with this repo

def load_and_prepare(path=CSV_PATH):
    """Load the cleaned flight+weather dataset and fix known quirks."""
    df = pd.read_csv(path)

    # The weather column headers were saved with a mis-encoded degree sign, e.g.
    # "Temperature (�F)_Avg". Strip any non-ASCII bytes so the names become
    # clean "(F)" labels, then collapse any doubled whitespace.
    df.columns = [re.sub(r"\s+", " ", re.sub(r"[^\x00-\x7F]", "", c)).strip()
                  for c in df.columns]

    # Parse the scheduled departure timestamp into a real datetime.
    df["Scheduled Time"] = pd.to_datetime(df["Departure_Scheduled"], errors="coerce")

    # Friendly name for the prediction target (minutes of departure delay).
    df = df.rename(columns={"Departure_Delay_Minutes": "Delay (minutes)"})

    # Drop the few rows without a usable timestamp or target.
    df = df.dropna(subset=["Scheduled Time", "Delay (minutes)"]).reset_index(drop=True)
    return df

df = load_and_prepare()
print("Loaded:", df.shape)
df.head()

In [ ]:

# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------
def assign_season(month):
    return {12: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1,
            6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3}[month]

def add_features(df):
    df = df.copy()

    # Composite "weather severity" index (wind + dryness + temperature swing).
    df["Weather Severity"] = (
        df["Wind Speed (mph)_Max"] * 0.4
        + (100 - df["Humidity (%)_Avg"]) * 0.3
        + (df["Temperature (F)_Max"] - df["Temperature (F)_Min"]) * 0.3
    )

    # Temporal features from the scheduled timestamp.
    hour = df["Scheduled Time"].dt.hour
    df["IsWeekend"] = (df["Scheduled Time"].dt.weekday >= 5).astype(int)
    df["PeakHour"] = (hour.between(6, 9) | hour.between(17, 20)).astype(int)
    df["Season"] = df["Scheduled Time"].dt.month.map(assign_season)
    df["Scheduled Hour"] = hour
    df["Scheduled Weekday"] = df["Scheduled Time"].dt.weekday
    df["Scheduled Month"] = df["Scheduled Time"].dt.month

    # Airport-level average delay (departure airport reputation).
    df["Avg Departure Delay (Airport)"] = (
        df.groupby("Departure_IATA")["Delay (minutes)"].transform("mean")
    )
    return df

df = add_features(df)

# Columns used by every model below.
NUMERIC_FEATURES = [
    "Temperature (F)_Max", "Temperature (F)_Avg", "Temperature (F)_Min",
    "Dew Point (F)_Max", "Dew Point (F)_Avg", "Dew Point (F)_Min",
    "Humidity (%)_Max", "Humidity (%)_Avg", "Humidity (%)_Min",
    "Wind Speed (mph)_Max", "Wind Speed (mph)_Avg", "Wind Speed (mph)_Min",
    "Pressure (in)_Max", "Pressure (in)_Avg", "Pressure (in)_Min",
    "Hour_of_Day", "Week_Number", "Scheduled Hour", "Scheduled Weekday",
    "Scheduled Month", "Weather Severity", "IsWeekend", "PeakHour", "Season",
    "Avg Departure Delay (Airport)",
]
CATEGORICAL_FEATURES = [
    "Departure_IATA", "Arrival_IATA", "Airline_IATA", "Day_of_Week", "Month_of_Year",
]

# Impute any residual missing values so the models never see NaNs.
for c in NUMERIC_FEATURES:
    df[c] = df[c].fillna(df[c].median())
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].fillna("unknown")

print("Numeric features :", len(NUMERIC_FEATURES))
print("Categorical      :", len(CATEGORICAL_FEATURES))
df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].head()

In [ ]:

# ---------------------------------------------------------------------------
# Reusable preprocessing pipeline: scale numbers + one-hot encode categories
# ---------------------------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
print("Design matrix:", X.shape)

## 2. Binary classification with hyper-parameter tuning

`GridSearchCV` over Logistic Regression and Random Forest. The best model's predictions are written to a submission file.

In [ ]:

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

y_binary = (df["Delay (minutes)"] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, stratify=y_binary, random_state=42)

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, class_weight="balanced"),
                            {"clf__C": [0.1, 1.0, 10.0]}),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                            {"clf__max_depth": [16, 20]}),
}

binary_results = {}
for name, (model, grid) in models.items():
    print(f"\n--- Tuning {name} ---")
    gs = GridSearchCV(Pipeline([("pre", make_preprocessor()), ("clf", model)]),
                      grid, cv=3, scoring="f1", n_jobs=-1)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    binary_results[name] = {
        "best_params": gs.best_params_,
        "estimator": gs.best_estimator_,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }
    print("Best params:", gs.best_params_)
    print(classification_report(y_test, y_pred, target_names=["On-Time", "Delayed"]))

# Pick the best model by F1 and export its predictions + the fitted model.
best_name = max(binary_results, key=lambda n: binary_results[n]["f1"])
best = binary_results[best_name]["estimator"]
print(f"\nBest binary model: {best_name} (F1={binary_results[best_name]['f1']:.4f})")

y_pred = best.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["On-Time", "Delayed"], yticklabels=["On-Time", "Delayed"])
plt.title(f"{best_name}  Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

submission = pd.DataFrame({
    "ID": range(1, len(y_pred) + 1),
    "Delay": ["delayed" if p == 1 else "on-time" for p in y_pred],
})
submission.to_csv("binary_classification_submission.csv", index=False)
joblib.dump(best, "best_binary_model.pkl")
print("Wrote binary_classification_submission.csv and best_binary_model.pkl")

## 3. Multi-class classification with tuning

In [ ]:

delay_bucket = pd.cut(
    df["Delay (minutes)"], bins=[-1, 0, 45, 175, float("inf")],
    labels=["No Delay", "Short Delay", "Moderate Delay", "Long Delay"],
).astype(str)
labels = sorted(delay_bucket.unique())
print("Class counts:\n", delay_bucket.value_counts(), "\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, delay_bucket, test_size=0.2, stratify=delay_bucket, random_state=42)

gs = GridSearchCV(
    Pipeline([("pre", make_preprocessor()),
              ("clf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))]),
    {"clf__max_depth": [16, 20]},
    cv=3, scoring="f1_macro", n_jobs=-1)
gs.fit(X_train, y_train)
y_pred = gs.predict(X_test)

print("Best params:", gs.best_params_)
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Macro F1 :", round(f1_score(y_test, y_pred, average="macro"), 4))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Multi-Class  Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

submission = pd.DataFrame({"ID": range(1, len(y_pred) + 1), "Delay": y_pred})
submission.to_csv("multiclass_classification_submission.csv", index=False)
print("Wrote multiclass_classification_submission.csv")

## 4. Regression  predicting exact delay in minutes

Several regressors are compared with cross-validated tuning; metrics are MAE, RMSE and R. The best model's predictions are exported.

In [ ]:

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_reg = df["Delay (minutes)"].astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

regressors = {
    "Linear Regression": (LinearRegression(), {}),
    "Ridge Regression":  (Ridge(random_state=42), {"clf__alpha": [0.1, 1.0, 10.0]}),
    "Lasso Regression":  (Lasso(random_state=42), {"clf__alpha": [0.01, 0.1, 1.0]}),
    "Random Forest":     (RandomForestRegressor(n_estimators=30, max_depth=10,
                                                random_state=42, n_jobs=-1), {}),
}

reg_results = {}
for name, (model, grid) in regressors.items():
    print(f"\n--- Tuning {name} ---")
    gs = GridSearchCV(Pipeline([("pre", make_preprocessor()), ("clf", model)]),
                      grid, cv=3, scoring="r2", n_jobs=-1)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    reg_results[name] = {
        "estimator": gs.best_estimator_,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
        "R2": r2_score(y_test, y_pred),
    }
    print("  MAE = {MAE:.3f}   RMSE = {RMSE:.3f}   R2 = {R2:.3f}".format(**reg_results[name]))

summary = pd.DataFrame(reg_results).T[["MAE", "RMSE", "R2"]].astype(float)
print("\nRegression comparison:\n", summary.round(3))

best_reg_name = summary["R2"].idxmax()
best_reg = reg_results[best_reg_name]["estimator"]
print(f"\nBest regressor: {best_reg_name}")

y_pred = best_reg.predict(X_test)
submission = pd.DataFrame({
    "ID": range(1, len(y_pred) + 1),
    "Predicted Delay (minutes)": np.round(np.clip(y_pred, 0, None), 2),
})
submission.to_csv("regression_submission.csv", index=False)
print("Wrote regression_submission.csv")

plt.figure(figsize=(8, 5))
summary["R2"].sort_values().plot(kind="barh", color="teal")
plt.title("Regression Model R Comparison")
plt.xlabel("R score")
plt.tight_layout()
plt.show()

---
**Phase 2 complete.** Submission files written:
- `binary_classification_submission.csv`
- `multiclass_classification_submission.csv`
- `regression_submission.csv`